In [ ]:
pip install datasets transformers sentence-transformers flet

In [ ]:
import flet as ft

import nest_asyncio
nest_asyncio.apply()

class TodoApp(ft.Column):
    # application's root control is a Column containing all other controls
    def __init__(self):
        super().__init__()
        self.new_task = ft.TextField(hint_text="What needs to be done?", expand=True)
        self.tasks_view = ft.Column()
        self.width = 600
        self.controls = [
            ft.Row(
                controls=[
                    self.new_task,
                    ft.FloatingActionButton(
                        icon=ft.Icons.ADD, on_click=self.add_clicked
                    ),
                ],
            ),
            self.tasks_view,
        ]

    def add_clicked(self, e):
        self.tasks_view.controls.append(ft.Checkbox(label=self.new_task.value))
        self.new_task.value = ""
        self.update()


def main(page: ft.Page):
    page.title = "To-Do App"
    page.horizontal_alignment = ft.CrossAxisAlignment.CENTER
    page.update()

    # create application instance
    todo = TodoApp()

    # add application's root control to the page
    page.add(todo)

ft.app(main)

Installing flet-web 0.27.6 package...OK


In [ ]:
from transformers import AutoTokenizer, AutoModel, T5ForConditionalGeneration # Import AutoModel instead of AutoModelForSeq2SeqLM
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer # Import SentenceTransformer

model = T5ForConditionalGeneration.from_pretrained("grammarly/coedit-large")
tokenizer = AutoTokenizer.from_pretrained("grammarly/coedit-large")

def get_embedding(text):
    """Get sentence embedding."""
    input_ids = tokenizer(text, return_tensors="pt").input_ids
    embedding = model.encoder(input_ids).last_hidden_state[:, 0, :].detach().numpy()

    return embedding # Return the embedding


def get_similarity(A, B):
    """Compute cosine similarity."""
    vec1, vec2 = get_embedding(A), get_embedding(B)
    # cosine_similarity expects 2D arrays, so reshape if necessary
    vec1 = vec1.reshape(1, -1)
    vec2 = vec2.reshape(1, -1)
    return cosine_similarity(vec1, vec2)[0][0] # Extract the similarity score

def generate_response(A):
  tokenizer = AutoTokenizer.from_pretrained("grammarly/coedit-large")
  model = T5ForConditionalGeneration.from_pretrained("grammarly/coedit-large")
  input_text = 'Rephrase to be polite: Yash you are an intelligent individual.'+A
  input_ids = tokenizer(input_text, return_tensors="pt").input_ids
  outputs = model.generate(input_ids, max_length=512)
  edited_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
  return edited_text



In [ ]:
class TreeNode:
    def __init__(self, message, frequency=0, probability=0, speaker=None):
        self.message = message
        self.frequency = frequency
        self.probability = probability
        self.speaker = speaker
        self.responses = []

def calculate_probability(node):
    if not node or not node.responses:
        return

    total_frequency = node.frequency

    if total_frequency == 0:
        return

    for child in node.responses:
        p = 0
        p=p+(child.frequency*child.probability)

    node.probability = p / total_frequency

    for child in node.responses:
        calculate_probability(child)

# Root node - Initial sales pitch
def build_salesbot():
    root = TreeNode(
        "Hi, I'm Alex from XYZ Insurance. We are currently offering insurance to our premium customers. Would you like to know more?",
        frequency=3000,
        speaker="Agent",
    )

    # Customer responses
    not_interested = TreeNode("Not interested", frequency= 3000, speaker="Customer",)
    root.responses.append(not_interested)

    # Sales responses to "Not Interested"
    limited_offer = TreeNode(
        "I can understand that you are not interested at the moment. You are 23 years old now and this is the lowest insurance premium that you will get. You are wise and you know what is best.",
        frequency= 600,
        speaker="Agent"
    )
    value_proposition = TreeNode(
        "I can understand that you are not interested in insurance at this moment. However, I believe that if I could give you some more details, you probably will be able to make a more informed judgment.",
        frequency= 2400,
        speaker="Agent"
    )

    not_interested.responses.extend([limited_offer, value_proposition])

    # Responses to the limited offer
    not_interested_limited_offer = TreeNode("The product unfortunately does not interest me. Take Care", frequency=540, speaker="Customer")
    why_matters_limited_offer = TreeNode("Even if the insurance is the cheapest for me, why would that matter?", frequency=60, speaker="Customer")
    limited_offer.responses.extend([not_interested_limited_offer, why_matters_limited_offer])

    # Responses to value proposition offer
    not_interested_value_proposition = TreeNode("The product unfortunately does not interest me. Take Care", frequency=2400, speaker="Customer")
    open_to_info_value_proposition = TreeNode("Sure, I’m open to hearing more information.", frequency=0, speaker="Customer")
    value_proposition.responses.extend([not_interested_value_proposition, open_to_info_value_proposition])

    # Responding to "why would that matter?"
    dreams_why_matters = TreeNode(
        "We all have dreams. That is all that matters in life. My dream is to build a villa next to the sea. For that I will need to work hard and take risks in life. What is your dream in life?",
        frequency=40,
        speaker="Agent"
    )
    risk_mitigant_why_matters = TreeNode(
        "Life is full of uncertainties—accidents, natural disasters, you name it. But imagine being able to protect your future with just one small monthly payment. The earlier you start, the more affordable it is.",
        frequency=20,
        speaker="Agent"
    )
    why_matters_limited_offer.responses.extend([dreams_why_matters, risk_mitigant_why_matters])

    # Risk Mitigant rejection
    not_interested_risk_mitigant = TreeNode("The product unfortunately does not interest me. Take Care", frequency=20, speaker="Customer")
    risk_mitigant_why_matters.responses.append(not_interested_risk_mitigant)

    # Responding to dreams
    billionaire_dreams = TreeNode("I want to work in Tech and someday lead a multi-million dollar tech start-up as a founder", frequency=32, speaker="Customer")
    simple_life_dreams = TreeNode("I want a simple life", frequency= 8, speaker="Customer", probability=0.0)
    dreams_why_matters.responses.extend([billionaire_dreams, simple_life_dreams])

    how_to_be_billionaire = TreeNode(
        "I really respect Start-up founders because of the amount of risks they take. Imagine putting all your money to your dream. It's a huge gamble sometimes. That's when I realize health is the most important asset for them. Have you heard about the 3rd Microsoft cofounder Ken Evans?",
        frequency=32,
        speaker="Agent"
    )
    billionaire_dreams.responses.append(how_to_be_billionaire)

    ken = TreeNode("Who was Ken?", frequency= 32, speaker="Customer")
    how_to_be_billionaire.responses.append(ken)

    response_to_ken = TreeNode(
        "Ken was the third cofounder of Microsoft. He unfortunately passed away during a trekking accident. He could never realize the value of what he was building. These kinds of stories tell us that while we follow our passion, we must protect what truly matters to us. It is always a great idea to receive a part of what could have been your earning potential, when disaster hits",
        frequency=32,
        speaker="Agent"
    )
    ken.responses.append(response_to_ken)

    emotional_about_ken = TreeNode(
        "Wow, that's really sad. It’s true—life is unpredictable. I never really thought about how unprotected I am. Let’s go ahead and set this up. What do you need from me?",
        frequency= 20,
        speaker="Customer",
        probability=1.0
    )
    logical_thought_about_ken = TreeNode(
        "You’re right. We work so hard chasing goals, but forget to safeguard what we already have. I think it’s time I got serious about insurance. Let’s do this.",
        frequency= 9,
        speaker="Customer",
        probability=1.0
    )
    rejection_after_ken = TreeNode(
        "That’s a powerful story, and I do see the value. But I’m just not in the right place to commit financially right now. Maybe later down the line.",
        frequency=3,
        speaker="Customer",
        probability=0.0
    )
    response_to_ken.responses.extend([rejection_after_ken, logical_thought_about_ken, emotional_about_ken])

    calculate_probability(root)
    return root

def print_conversation_log(node, level=0):
    if node is None:
        return

    print(f"{node.speaker}: {node.message} probability {node.probability}")

    if not node.responses:
        print("  " * (level + 1) + "-- End of conversation --")
        return

    if node.speaker.lower() == "agent":
        user_input = input(f"Your input to agent: ")
        best_match = max(node.responses, key=lambda x: get_similarity(user_input, x.message))
        best_match.frequency += 1
        print_conversation_log(best_match, level + 1)

    elif node.speaker.lower() == "customer":
        best_response = max(node.responses, key=lambda x: x.probability)
        best_response.frequency += 1
        print(f"Agent's response: {generate_response(best_response.message)}")
        print(f"Probability:{best_response.probability}")
        print_conversation_log(best_response, level + 1)

# Run it
root = build_salesbot()
print_conversation_log(root)
